# 6. CHURN ATTRIBUTION AND EXPLAINABILITY
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../models/churn_model_YYYYMMDD.joblib`, `../data/processed/churn_features_YYYYMMDD.parquet`, and `../data/processed/churn_predictions_YYYYMMDD.parquet`

*The trained XGBoost model, the modeling matrix, and the scored customer snapshots from NB04.*

**OUTPUT:** `../data/processed/churn_explainability_YYYYMMDD.parquet`, `../data/processed/churn_driver_summary_YYYYMMDD.csv`, and `../reports/churn_explainability_YYYYMMDD.html`

*A SHAP-based explainability package connecting churn predictions to actionable drivers, customer profiles, and retention decisions.*


---
## 6.1. STARTING SITUATION


NB05 established that the model ranking is usable for campaign prioritization. The next business question is not only *who* is high risk, but *why* each customer is high risk and what kind of action should be triggered.

This notebook therefore turns model scores into **interpretable churn drivers**. It uses SHAP to quantify the factors pushing risk upward or downward and maps those drivers into the retention-action framework defined for VivaMarket Brasil.


---
## 6.2. NOTEBOOK OBJECTIVE


- **Business objective:** connect churn predictions to concrete retention actions such as win-back, logistics apology, category reactivation, or VIP escalation.
- **Analytical objective:** compute SHAP-based global and local explanations, summarize dominant drivers by risk tier, and prepare explainability outputs for orchestration and reporting.


In [1]:
import base64
import io
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb06_explainability')
logger.info('NB06 started: churn attribution and explainability.')

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


2026-05-02 00:38:30,686 | INFO | NB06 started: churn attribution and explainability.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

run_date_tag = datetime.now(ZoneInfo('Europe/Paris')).strftime('%Y%m%d')
model_path = sorted(MODELS_DIR.glob('churn_model_*.joblib'))[-1]
feature_path = sorted(PROCESSED_DIR.glob('churn_features_*.parquet'))[-1]
prediction_path = sorted(PROCESSED_DIR.glob('churn_predictions_*.parquet'))[-1]
explainability_path = PROCESSED_DIR / f'churn_explainability_{run_date_tag}.parquet'
driver_summary_path = PROCESSED_DIR / f'churn_driver_summary_{run_date_tag}.csv'
explainability_html_path = REPORTS_DIR / f'churn_explainability_{run_date_tag}.html'


In [3]:
package = joblib.load(model_path)
model = package['model']
feature_columns = package['feature_columns']
feature_df = pd.read_parquet(feature_path)
feature_df['snapshot_date'] = pd.to_datetime(feature_df['snapshot_date'])
prediction_df = pd.read_parquet(prediction_path)
prediction_df['snapshot_date'] = pd.to_datetime(prediction_df['snapshot_date'])

leakage_columns = [
    'customer_unique_id', 'snapshot_key', 'snapshot_date', 'first_purchase_timestamp',
    'last_purchase_timestamp', 'future_orders_90d', 'future_revenue_90d', 'churn_90d_label'
]
test_df = feature_df[feature_df['snapshot_key'].isin(package['test_snapshot_keys'])].copy().reset_index().rename(columns={'index': 'source_row_id'})
X_test = pd.get_dummies(
    test_df[[c for c in feature_df.columns if c not in leakage_columns]],
    columns=['customer_state'],
    dtype=float,
).reindex(columns=feature_columns, fill_value=0.0)
X_test.index = test_df['source_row_id']

explainability_base = prediction_df.merge(
    test_df[[
        'source_row_id', 'customer_unique_id', 'snapshot_key', 'snapshot_date', 'customer_state',
        'avg_review_score', 'late_delivery_rate_total', 'revenue_90d', 'credit_card_share_total',
        'boleto_share_total', 'voucher_share_total'
    ]],
    on=['customer_unique_id', 'snapshot_key', 'snapshot_date'],
    how='left',
    validate='one_to_one',
).set_index('source_row_id')
explainability_base.head()


,customer_unique_id,snapshot_key,snapshot_date,recency_days,total_orders,total_payment_value,orders_30d,orders_90d,churn_90d_label,churn_probability,risk_tier,selected_model,customer_state,avg_review_score,late_delivery_rate_total,revenue_90d,credit_card_share_total,boleto_share_total,voucher_share_total
source_row_id,,,,,,,,,,,,,,,,,,,
168908,0004bd2a26a76fe21f786e4fbd80607f,20180501,2018-05-01,26,1,166.9800,1.0000,1,1,0.5585,MEDIUM,xgboost,SP,4.0000,0.0000,166.9800,1.0000,0.0000,0.0000
168909,00050ab1314c0e55a6ca13cf7181fecf,20180501,2018-05-01,11,1,35.3800,1.0000,1,1,0.5727,MEDIUM,xgboost,SP,4.0000,0.0000,35.3800,0.0000,1.0000,0.0000
168910,00053a61a98854899e70ed204dd4bafe,20180501,2018-05-01,62,1,419.1800,0.0000,1,1,0.8962,HIGH,xgboost,PR,1.0000,0.0000,419.1800,1.0000,0.0000,0.0000
168911,0005ef4cd20d2893f0d9fbd94d3c0d97,20180501,2018-05-01,50,1,129.7600,0.0000,1,1,0.7706,HIGH,xgboost,MA,1.0000,1.0000,129.7600,1.0000,0.0000,0.0000
168912,00090324bbad0e9342388303bb71ba0a,20180501,2018-05-01,38,1,63.6600,0.0000,1,1,0.7081,HIGH,xgboost,SP,5.0000,0.0000,63.6600,1.0000,0.0000,0.0000


In [4]:
sample_n = min(5000, len(X_test))
X_sample = X_test.sample(n=sample_n, random_state=42).sort_index()
base_sample = explainability_base.loc[X_sample.index].copy()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)
shap_array = np.asarray(shap_values)
logger.info('SHAP matrix shape: %s', shap_array.shape)

mean_abs_shap = np.abs(shap_array).mean(axis=0)
global_importance = (
    pd.DataFrame({'feature': X_sample.columns, 'mean_abs_shap': mean_abs_shap})
    .sort_values('mean_abs_shap', ascending=False)
    .reset_index(drop=True)
)
global_importance.head(15)


2026-05-02 00:38:32,732 | INFO | SHAP matrix shape: (5000, 111)


,feature,mean_abs_shap
0,recency_days,0.1547
1,total_freight_value,0.1303
2,max_installments,0.1129
3,total_item_price,0.0935
4,avg_review_score,0.0810
5,avg_order_value,0.0617
6,avg_order_value_90d,0.0613
7,revenue_90d,0.0606
8,credit_card_value_total,0.0598
9,credit_card_value_90d,0.0544


In [5]:
def map_driver(feature_name: str) -> str:
    name = feature_name.lower()
    if 'recency' in name or 'active_last' in name:
        return 'recency'
    if 'late_delivery' in name or 'review' in name:
        return 'logistics'
    if 'frequency' in name or 'orders_' in name or 'mean_gap' in name:
        return 'frequency'
    if 'revenue' in name or 'payment' in name or 'monetary' in name or 'value' in name:
        return 'monetary'
    if 'category' in name or 'product' in name:
        return 'category'
    return 'other'

top_idx = np.abs(shap_array).argmax(axis=1)
top_features = [X_sample.columns[i] for i in top_idx]
top_feature_shap = shap_array[np.arange(len(X_sample)), top_idx]

explainability_sample = base_sample.copy()
explainability_sample['top_shap_feature'] = top_features
explainability_sample['top_shap_value'] = top_feature_shap
explainability_sample['top_driver_group'] = [map_driver(name) for name in top_features]
explainability_sample['ltv_segment'] = pd.qcut(
    explainability_sample['total_payment_value'].rank(method='first'),
    q=4,
    labels=['ENTRY', 'CORE', 'GROWTH', 'VIP']
)

explainability_sample['recommended_offer_type'] = np.select(
    [
        explainability_sample['top_driver_group'].eq('recency'),
        explainability_sample['top_driver_group'].eq('frequency'),
        explainability_sample['top_driver_group'].eq('logistics'),
        explainability_sample['top_driver_group'].eq('category'),
    ],
    [
        'reactivation_urgency',
        'repeat_purchase_nurturing',
        'logistics_apology_priority_shipping',
        'category_specific_winback',
    ],
    default='value_bundle_offer',
)
explainability_sample['recommended_discount_pct'] = np.select(
    [
        explainability_sample['risk_tier'].eq('HIGH') & explainability_sample['ltv_segment'].eq('VIP'),
        explainability_sample['risk_tier'].eq('HIGH'),
        explainability_sample['risk_tier'].eq('MEDIUM'),
    ],
    [30, 25, 12],
    default=0,
)
explainability_sample['free_shipping_flag'] = np.where(explainability_sample['risk_tier'].isin(['HIGH', 'MEDIUM']), True, False)
explainability_sample['vip_human_touch_flag'] = np.where(
    explainability_sample['risk_tier'].eq('HIGH') & explainability_sample['ltv_segment'].eq('VIP'), True, False
)
explainability_sample.head()


,customer_unique_id,snapshot_key,snapshot_date,recency_days,total_orders,total_payment_value,orders_30d,orders_90d,churn_90d_label,churn_probability,risk_tier,selected_model,customer_state,avg_review_score,late_delivery_rate_total,revenue_90d,credit_card_share_total,boleto_share_total,voucher_share_total,top_shap_feature,top_shap_value,top_driver_group,ltv_segment,recommended_offer_type,recommended_discount_pct,free_shipping_flag,vip_human_touch_flag
source_row_id,,,,,,,,,,,,,,,,,,,,,,,,,,,
168914,0010fb34b966d44409382af9e8fd5b77,20180501,2018-05-01,57,1,61.8000,0.0000,1,1,0.6982,MEDIUM,xgboost,SP,4.0000,1.0000,61.8000,1.0000,0.0000,0.0000,avg_order_value_90d,0.1876,monetary,ENTRY,value_bundle_offer,12,True,False
168942,00737247e3dc1942da2c527ef3bd2709,20180501,2018-05-01,38,1,78.2000,0.0000,1,1,0.7669,HIGH,xgboost,MG,1.0000,1.0000,78.2000,1.0000,0.0000,0.0000,max_installments,0.1454,other,CORE,value_bundle_offer,25,True,False
168943,00762eafc5dac5fc1574f91e9b7c9b67,20180501,2018-05-01,83,1,233.9300,0.0000,1,1,0.6488,MEDIUM,xgboost,SP,5.0000,0.0000,233.9300,1.0000,0.0000,0.0000,recency_days,0.3937,recency,VIP,reactivation_urgency,12,True,False
168947,007e21b7a007c8f8dadfa43c9d5a91db,20180501,2018-05-01,52,1,70.7500,0.0000,1,1,0.4997,MEDIUM,xgboost,RJ,1.0000,1.0000,70.7500,0.0000,1.0000,0.0000,max_installments,0.1437,other,CORE,value_bundle_offer,12,True,False
168962,00b21d737fc7f475c145ca292832385d,20180501,2018-05-01,67,1,177.5900,0.0000,1,1,0.5827,MEDIUM,xgboost,MG,5.0000,0.0000,177.5900,1.0000,0.0000,0.0000,recency_days,0.1831,recency,GROWTH,reactivation_urgency,12,True,False


In [6]:
driver_summary = (
    explainability_sample.groupby(['risk_tier', 'top_driver_group', 'recommended_offer_type'], observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        avg_probability=('churn_probability', 'mean'),
        observed_churn_rate=('churn_90d_label', 'mean'),
        avg_discount_pct=('recommended_discount_pct', 'mean'),
    )
    .reset_index()
    .sort_values(['risk_tier', 'rows_n'], ascending=[True, False])
)
driver_summary.to_csv(driver_summary_path, index=False)
explainability_sample.to_parquet(explainability_path, index=False)
logger.info('Explainability parquet saved to %s', explainability_path)
logger.info('Driver summary saved to %s', driver_summary_path)
driver_summary.head(12)


2026-05-02 00:38:32,803 | INFO | Explainability parquet saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_explainability_20260502.parquet


2026-05-02 00:38:32,804 | INFO | Driver summary saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_driver_summary_20260502.csv


,risk_tier,top_driver_group,recommended_offer_type,rows_n,avg_probability,observed_churn_rate,avg_discount_pct
3,LOW,other,value_bundle_offer,138,0.2905,0.9855,0.0000
2,LOW,monetary,value_bundle_offer,101,0.3449,0.9901,0.0000
4,LOW,recency,reactivation_urgency,50,0.3519,0.9800,0.0000
1,LOW,logistics,logistics_apology_priority_shipping,1,0.3490,1.0000,0.0000
0,LOW,category,category_specific_winback,0,NaN,NaN,NaN
7,MEDIUM,monetary,value_bundle_offer,1142,0.5839,0.9886,12.0000
8,MEDIUM,other,value_bundle_offer,989,0.5760,0.9939,12.0000
9,MEDIUM,recency,reactivation_urgency,870,0.5801,0.9954,12.0000
6,MEDIUM,logistics,logistics_apology_priority_shipping,137,0.5943,1.0000,12.0000
5,MEDIUM,category,category_specific_winback,11,0.5671,1.0000,12.0000


In [7]:
def figure_to_base64(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', bbox_inches='tight', dpi=160)
    plt.close(fig)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=global_importance.head(12), x='mean_abs_shap', y='feature', ax=axes[0], color='#d62728')
axes[0].set_title('GLOBAL SHAP IMPORTANCE')

plot_driver = (
    explainability_sample['top_driver_group']
    .value_counts(normalize=True)
    .rename_axis('driver')
    .reset_index(name='share')
)
sns.barplot(data=plot_driver, x='share', y='driver', ax=axes[1], color='#1f77b4')
axes[1].set_title('TOP DRIVER MIX IN THE SCORING SAMPLE')
plt.tight_layout()
chart_b64 = figure_to_base64(fig)

html_parts = [
    '<html><head><meta charset="utf-8"><title>Churn Explainability</title></head><body>',
    '<h1>CHURN EXPLAINABILITY REPORT</h1>',
    '<h2>Global SHAP importance</h2>', global_importance.head(25).to_html(index=False),
    '<h2>Driver summary by risk tier</h2>', driver_summary.to_html(index=False),
    '<h2>Sample customer action table</h2>', explainability_sample.head(25).to_html(index=False),
    f'<h2>Visual summary</h2><img src="data:image/png;base64,{chart_b64}" style="max-width:1100px;">',
    '</body></html>'
]
explainability_html_path.write_text('\n'.join(html_parts), encoding='utf-8')
logger.info('Explainability report saved to %s', explainability_html_path)


2026-05-02 00:38:33,103 | INFO | Explainability report saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/churn_explainability_20260502.html


---
## 6.3. NOTEBOOK CLOSURE


The explainability layer now makes the churn model operationally interpretable. Instead of sending one generic campaign to every high-risk customer, VivaMarket can differentiate between inactivity-led churn, logistics frustration, low purchase frequency, or category-specific disengagement.

The next notebook should package that logic into a reusable scoring flow so the same rules can be executed consistently during daily inference.
